In [1]:
#import seaborn as sns
import pandas as pd
from pylab import *
%matplotlib inline
#from scipy.stats import gaussian_kde
from matplotlib.colors import ListedColormap
#from scipy.spatial import ConvexHull
#import statistics
#from scipy.spatial.distance import mahalanobis
from preprocessing_aux import *

In [2]:
infile = 'dataset_file.csv'
outfile = 'dataset_file_transformed.csv'
df = pd.read_csv(infile)

In [3]:
df['rc'] = df['R'] / (df['R'] + df['G'] + df['B'])
df['gc'] = df['G'] / (df['R'] + df['G'] + df['B'])

In [4]:
# --------------------------------------------------------------------------------------
# extending pada data frame by rc/gc, rc+gc, luminance Y, perceived lightness L, z_score_Y, z_score_L 
# Excess indecies ExG, ExR, ExB, indices ExGmExR, Ikaw, MGRVI, and GLI 
# --------------------------------------------------------------------------------------
# blue chromaticity
df['bc'] = 1-(df['rc']+df['gc'])

# ration rc/gc
df['rc/gc'] = df['rc']/df['gc']

# sum rc+gc
df['rc+gc'] = df['rc']+df['gc']

# excess indecies
ExG = [excess_index(G=val, R=np.array(df['R'])[i], B=np.array(df['B'])[i], convert_to_255=False)
       for i, val in enumerate(np.array(df['G']))]
df['ExG'] = ExG

ExR = [excess_index(G=val, R=np.array(df['R'])[i], B=np.array(df['B'])[i], convert_to_255=False, excessband='red')
       for i, val in enumerate(np.array(df['G']))]
df['ExR'] = ExR

ExB = [excess_index(G=val, R=np.array(df['R'])[i], B=np.array(df['B'])[i], convert_to_255=False, excessband='blue')
       for i, val in enumerate(np.array(df['G']))]
df['ExB'] = ExB

df.head()

# index ExGmExR
df['ExGmExR'] = df['ExG']-df['ExR']

# index Ikaw - Kawashima Index
df['Ikaw'] = (df['R']-df['B'])/(df['R']+df['B'])

# MGRVI - modified green red vegetation index
df['MGRVI'] = ((df['G'])**2 - (df['R'])**2)/((df['G'])**2 + (df['R'])**2)

# GLI - green leaf index
df['GLI'] = (2*df['G'] - df['R'] - df['B'])/(2*df['G'] + df['R'] + df['B'])

# luminance Y
Y = [0.2126*sRGBtoLin(val) + 
     0.7152*sRGBtoLin(np.array(df['G'])[i]) + 
     0.0722*sRGBtoLin(np.array(df['B'])[i]) for i, val in enumerate(np.array(df['R']))]
df['Y'] = Y

# perceived lightness L
L = [YtoLstar(val) for val in df['Y']]
df['L'] = L

# z_score of the luminance Y
df['z_score_Y'] = 0
sites = set(df['site'])
for site in sites:
    condition = df['site'] == site
    subdata_site = df[df['site'].isin([site])]['Y']
    z_site = z_score(x_all=np.array(subdata_site))
    df.loc[condition, 'z_score_Y'] = z_site
    
# z_score of the perceived lightness L
df['z_score_L'] = 0
sites = set(df['site'])
for site in sites:
    condition = df['site'] == site
    subdata_site = df[df['site'].isin([site])]['L']
    z_site = z_score(x_all=np.array(subdata_site))
    df.loc[condition, 'z_score_L'] = z_site

print(df.head())

     R    G    B  chm  veg_class    site  chicken_0  chicken_1  chicken_2   
0  223  228  245  223         13  F3-20C  -0.336462  -0.577480  -0.300406  \
1  190  198  217  190         13  F3-20C  -0.336462  -0.577480  -0.300406   
2  164  161  182  164         13  F3-20C  -0.336462  -0.577480  -0.300406   
3  207  215  233  207         13  F3-20C  -0.336462  -0.577480  -0.300406   
4  171  179  160  171          6  F3-20C  -0.210313  -0.434953   0.897604   

   chicken_3  ...   ExR    ExB  ExGmExR      Ikaw     MGRVI       GLI   
0  -0.397835  ...  84.2  115.0    -96.2 -0.047009  0.022170 -0.012987  \
1  -0.397835  ...  68.0  105.8    -79.0 -0.066339  0.041220 -0.013699   
2  -0.397835  ...  68.6   93.8    -92.6 -0.052023 -0.018460 -0.035928   
3  -0.397835  ...  74.8  111.2    -84.8 -0.059091  0.037901 -0.011494   
4  -0.080942  ...  60.4   45.0    -33.4  0.033233  0.045690  0.039187   

               Y            L  z_score_Y  z_score_L  
0  402354.630352  8547.691039   1.075073   0

In [5]:
df.to_csv(outfile, index=False)